# Predicción del fútbol uruguayo: selección temporal de modelos

Se conserva la explicación del procesamiento, ID3 y Naive Bayes propio, y se
comparan los cuatro métodos requeridos junto con los árboles de referencia y
el baseline. Esta etapa llega hasta **validación 2021–2023**. La evaluación
2024–2025 está desactivada por defecto (`RUN_FINAL_TEST = False`).

La ejecución reproducible de las grillas está en `scripts/run_validation.py`.
Este notebook carga las métricas guardadas, comprueba su procedencia y muestra
las curvas sin repetir entrenamientos. Los CSV y el manifiesto están en
`results/validation/`; el análisis está en `docs/validation_findings.md`.


## 1. Configuración

Entorno de la consigna: Python 3.12 y scikit-learn 1.9. Los módulos de `src/`
comparten la política de datos, historiales y validación. La semilla es 42.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

ROOT = Path.cwd()
if not (ROOT / 'requirements.txt').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from features import (  # noqa: E402
    NUMERIC_FEATURES,
    build_causal_match_features,
    load_clean_matches,
)
from baseline import TenYearWinRateClassifier  # noqa: E402
from id3 import ID3  # noqa: E402
from naive_bayes import MEstimateCategoricalNB  # noqa: E402
from preprocessing import MixedTypeDiscretizer  # noqa: E402
from baseline import BASELINE_FEATURES  # noqa: E402
from evaluation import (  # noqa: E402
    temporal_holdout, make_temporal_folds,
    new_discretizer, evaluate_temporal_cv,
)

RAW = ROOT / 'data/raw/futbol_uruguayo.zip'
CLASSES = ['E', 'L', 'V']
BIN_ORDER = ['baja', 'media', 'alta']
RANDOM_STATE = 42
pd.set_option('display.max_columns', 30)


def entropia(series: pd.Series) -> float:
    """Entropia de Shannon en bits de una serie de etiquetas."""
    probs = series.value_counts(normalize=True)
    return float(-(probs * np.log2(probs)).sum())

## 2. Datos: lectura, auditoría y limpieza del ZIP

`load_clean_matches` aplica la política de `docs/data_policy.md` antes de crear
etiquetas o historiales: elimina repeticiones exactas, pone en cuarentena todas
las variantes de una fecha/par de equipos ambiguo (incluida localía invertida)
y excluye los registros `full_time=E/P`, cuyos goles a los 90 minutos no están
separados. No elige un marcador entre variantes ni reconstruye goles.

En los registros `F` conservados, `winner` es `L` si `gh > ga`, `V` si `gh < ga`
y `E` si son iguales. `full_time=E` significa prórroga, no la etiqueta empate.
Los goles del partido no son entradas. La auditoría reproducible, con líneas
del CSV y alertas adicionales de equipo/fecha, está en `docs/data_audit/`.


In [ ]:
matches = load_clean_matches(RAW)
# Separar el horizonte ANTES de construir atributos o consultar distribuciones.
matches = matches.loc[matches['date'].dt.year.le(2023)].copy()
print('Filas limpias:', len(matches))
display(matches.head(3))
display(matches.dtypes.to_frame(name='tipo'))
print('Distribucion de la clase (hasta 2023):')
display(
    matches['winner']
    .value_counts()
    .reindex(CLASSES)
    .rename('cantidad')
    .to_frame()
)

## 3. Atributos causales (sin mirar el futuro)

`build_causal_match_features` construye, para cada partido, tasas historicas calculadas **con fechas estrictamente anteriores**. Los partidos de un mismo dia se featurizan todos antes de actualizar los historiales, de modo que un partido jamas usa el resultado de otro partido del mismo dia (esto evita **leakage**).

El modelo usa 6 tasas de victoria:

- `home/away_win_rate_last_5`: forma de cada equipo en sus ultimos 5 partidos;
- `home/away_win_rate_season`: exito de cada equipo dentro del ano calendario;
- `home_win_rate_as_home_all`: historico del local actuando de local;
- `home_win_rate_h2h_as_home`: historico del local contra ese visitante con la misma localia (head-to-head orientado).

Cuando un equipo **no tiene historial** (debut, primera vez en la ventana o primer partido del ano) el denominador es 0 y se imputa un neutro de `0.5` en lugar de `0.0`, para separarlo de cero victorias; el neutro puede coincidir con una tasa real de 0.5.

El baseline usa las tasas causales de diez años del **mismo constructor**, con
los mismos partidos admitidos y el mismo valor neutro. Todos los modelos
pueden usar resultados de fechas anteriores de validación/test; ninguno usa
resultados de la fecha actual y ninguno se reajusta dentro de ese bloque.

In [ ]:
featured = build_causal_match_features(matches)
print('Partidos con atributos:', len(featured))
display(featured[['date', 'home', 'away'] + NUMERIC_FEATURES + ['winner']].head(5))

sin_historial = featured[featured['home_win_rate_season'].eq(0.5)]
print('Tasa 0.5: puede ser imputada o una proporción real, no prueba ausencia de historial:')
display(sin_historial[['date', 'home', 'away'] + NUMERIC_FEATURES].head(3))

## 4. Partición temporal y folds comunes

- Entrenamiento final: hasta 2023 inclusive.
- Evaluación reservada: 2024–2025.
- Validación: años completos 2021, 2022 y 2023, entrenando con todos los años
  anteriores a cada bloque. No se separan partidos de una misma fecha.

`CV_FOLDS` es la única definición de folds para ID3, NB, baseline y comparadores.
Cada llamada a `evaluate_temporal_cv` ajusta un pipeline nuevo por fold, con
selección de entradas y, cuando corresponde, discretización aprendida solo en
ese entrenamiento. Las métricas de test no seleccionan parámetros.


In [ ]:
train = featured.reset_index(drop=True)
assert train['date'].dt.year.le(2023).all()
CV_FOLDS = make_temporal_folds(train)
print('Partidos disponibles para selección:', len(train))
print('No se construyen atributos ni se evalúa el período 2024–2025.')
print('Folds compartidos:')
display(pd.DataFrame([
    {'validacion': fold.validation_year,
     'train_hasta': train.iloc[list(fold.train_positions)]['date'].max().date(),
     'validacion_desde': train.iloc[list(fold.validation_positions)]['date'].min().date(),
     'validacion_hasta': train.iloc[list(fold.validation_positions)]['date'].max().date(),
     'n_train': len(fold.train_positions),
     'n_validacion': len(fold.validation_positions)}
    for fold in CV_FOLDS
]))


## 5. Discretización

ID3 y Naive Bayes categórico reciben códigos enteros. Las tasas de los últimos
cinco partidos usan cortes fijos `[0.3, 0.6]`; con menos de cinco antecedentes
se calcula la proporción sobre los disponibles. Las otras tasas usan terciles
aprendidos en entrenamiento. Los valores repetidos pueden producir tamaños
desiguales o menos de tres intervalos. No hay categoría exclusiva sin historial.

Aquí se prepara la transformación final con todo train; **no** se reutiliza en
validación cruzada, que crea su propio discretizador dentro de cada fold.


In [ ]:
discretizer = new_discretizer()
discretizer.fit(train[NUMERIC_FEATURES])

print('Bordes de cada atributo (dos cortes -> tres bines):')
display(
    pd.DataFrame(
        {column: discretizer.numeric_edges_[column] for column in NUMERIC_FEATURES}
    ).T.rename(columns={0: 'borde_1', 1: 'borde_2'})
)

X_train = discretizer.transform(train[NUMERIC_FEATURES])

print('Ejemplo: la tasa cruda y el codigo que recibe ID3 (1=baja, 2=media, 3=alta):')
muestra = train[['home', 'away'] + NUMERIC_FEATURES].head(5).copy()
all_columns = [column for column in NUMERIC_FEATURES]
codigos = pd.DataFrame(
    X_train[:5], columns=[column + '_cod' for column in NUMERIC_FEATURES]
)
display(pd.concat([muestra, codigos], axis=1))

print('Cuantas filas de train cae en cada categoria:')
resumen = pd.DataFrame({column: pd.Series(X_train[:, i]).map(
    {1: 'baja', 2: 'media', 3: 'alta'}
).value_counts() for i, column in enumerate(NUMERIC_FEATURES)})
display(resumen.reindex(BIN_ORDER).fillna(0).astype(int))

## 6. Como decide el arbol: entropia y ganancia

ID3 elige el atributo que mas reduce la **entropia** (desorden) de la clase. La entropia de la clase en train es:

```text
H(Y) = -sum_c P(c) * log2(P(c))
```

que mide cuantos bits de informacion hacen falta para decir el resultado sabiendo solo la distribucion global. La **ganancia de informacion** de un atributo es la entropia menos la entropia condicional promedio tras partir por ese atributo.

En este paso calculamos **la primera decision que tomaria el arbol** mostrando la ganancia de cada atributo sobre todo train.

In [ ]:
parent_entropy = entropia(train['winner'])
print(f'Entropia de la clase en train: {parent_entropy:.4f} bits '
      '(maximo teorico ~1.585 bits para 3 clases equiprobables)')

gains = []
for index, column in enumerate(NUMERIC_FEATURES):
    conditional = 0.0
    for code in np.unique(X_train[:, index]):
        mask = X_train[:, index] == code
        conditional += mask.mean() * entropia(train['winner'][mask])
    gains.append({
        'atributo': column,
        'entropia_condicional': round(conditional, 4),
        'ganancia': round(parent_entropy - conditional, 4),
    })
gains_frame = pd.DataFrame(gains).sort_values('ganancia', ascending=False)
display(gains_frame.reset_index(drop=True))

mejor = gains_frame.iloc[0]
print(
    f'Primera division sobre {mejor["atributo"]} '
    f'(ganancia {mejor["ganancia"]:.4f} bits).'
)

## 7. Naive Bayes propio

Ahora usamos `MEstimateCategoricalNB` (`src/naive_bayes.py`) con los mismos partidos, las seis tasas de `NUMERIC_FEATURES` y las clases `E`, `L`, `V`. Naive Bayes combina la frecuencia de cada clase con las probabilidades de los atributos y supone que estos son **independientes entre si dada la clase**.

Esta implementacion es **categorica**: requiere entradas discretas, codificadas como enteros no negativos. Reutilizamos la configuracion de `MixedTypeDiscretizer` de la seccion 5: tres bines, cortes fijos para las tasas de los ultimos cinco partidos y cuantiles para las restantes. El discretizador existente se ajusto solo con train; para validar necesitamos instancias nuevas ajustadas solo con el entrenamiento de cada fold.

El suavizado **m-estimate** evita probabilidades condicionales nulas:

\[
P(X_j=v \mid Y=c) = \frac{n_{jvc} + m\,p_{jv}}{n_c + m}
\]

Aqui `n_jvc` cuenta los partidos de clase `c` cuyo atributo `j` tiene codigo `v`, y `n_c` cuenta los partidos de esa clase. El prior `p_jv = 1 / K_j` es uniforme sobre los codigos posibles: en esta implementacion, desde `0` hasta el maximo codigo observado en entrenamiento, incluido el `0` reservado para desconocidos. `m > 0` es el peso total del prior: cuanto mayor es, mas se acercan las condicionales a la distribucion uniforme.

**Como leer `src/naive_bayes.py`, paso a paso:**

1. **`_validate_X`** comprueba que cada fila sea un partido y que los valores sean codigos enteros no negativos. Las tasas se discretizan antes de entrar al modelo; los codigos son categorias, no cantidades.
2. **`fit(X, y)`** aprende dos cosas: la frecuencia de cada resultado `P(c)` y una tabla por atributo con `P(codigo | c)`. `np.unique` convierte etiquetas en indices y `np.bincount` cuenta apariciones. En cada tabla, las filas son clases y las columnas son codigos.
3. **Suavizado:** si hay 10 partidos de clase `L`, un codigo aparece en 3 y hay 4 codigos posibles (incluido el `0`), con `m=2` la probabilidad es `(3 + 2/4) / (10 + 2) = 0.292`. Incluso un codigo sin apariciones recibe algo de probabilidad. Este suavizado se aplica a las tablas de atributos, no a `P(c)`.
4. **`_joint_log_likelihood`** puntua cada clase para un partido: parte de `log P(c)` y suma el log de la probabilidad de cada uno de sus seis codigos. Asi implementa el producto de Naive Bayes sin multiplicar numeros muy pequenos:

\[
\mathrm{puntaje}(c) = \log P(c) + \sum_j \log P(X_j=x_j \mid c).
\]

5. **Las salidas:** `predict` elige la clase de mayor puntaje; `predict_proba` usa `softmax` para convertir los puntajes en probabilidades que suman 1; `predict_log_proba` devuelve sus logaritmos. Las columnas siguen el orden de `classes_`. Si llega un codigo fuera del rango aprendido, se usa el codigo reservado `0`.

Por ejemplo, para `[3, 1, 2, 2, 3, 1]`, se busca cada codigo en la tabla de su atributo y se suman sus aportes para `E`, `L` y `V`. No se suma el valor numerico de los codigos: se suman los **logs de sus probabilidades**. Para elegir la clase no hace falta calcular `P(X)`, porque es el mismo divisor para las tres.


## 8. Grillas y regla de selección

Los folds son los mismos años completos 2021, 2022 y 2023. Cada combinación
ajusta un pipeline nuevo por fold, sin usar los cuantiles de otro entrenamiento.
Se selecciona por **macro-F1 medio de validación**, con igual peso por año y
las clases fijas E/L/V. El error es `1 - accuracy`; el de entrenamiento es de
resustitución y sirve como diagnóstico, no como estimación de generalización.

- ID3: `min_info_gain = [0, 0.001, 0.005, 0.01, 0.02, 0.05]`.
- NB propio: `m = [0.1, 1, 10, 100, 1000]`.
- CategoricalNB: `alpha = [0.025, 0.25, 2.5, 25, 250]`, `min_categories=4`.
- Random Forest: `max_depth = [4, 8, None]` y
  `min_samples_leaf = [50, 20, 5, 1]`; 300 árboles, semilla 42,
  `class_weight='balanced_subsample'`.

Ante empate exacto se elige el menor umbral/m/alpha; para el bosque, la menor
profundidad y luego la mayor hoja. Se conserva el bosque sin límite/hoja 1,
los dos DecisionTree con entropía (códigos y tasas) y el baseline de diez años.

Con cuatro códigos por atributo, el suavizado uniforme propio equivale a
`alpha=m/4`. Las grillas de ambos NB permiten comparar esa correspondencia;
no se espera que dos implementaciones de la misma fórmula sean necesariamente
dos métodos estadísticos distintos. El manifiesto registra las cardinalidades
observadas de cada fold. No se realizan experimentos con atributos nuevos.

Para reproducir la búsqueda (solo entrenamiento/validación):
`python3.12 scripts/run_validation.py`.


In [ ]:
import hashlib
import json

VALIDATION_DIR = ROOT / 'results/validation'
manifest = json.loads((VALIDATION_DIR / 'manifest.json').read_text())
assert manifest['source_sha256'] == hashlib.sha256(RAW.read_bytes()).hexdigest()
for relative, expected in manifest['implementation_sha256'].items():
    assert hashlib.sha256((ROOT / relative).read_bytes()).hexdigest() == expected, relative
assert manifest['test_evaluated'] is False
assert manifest['final_refit_performed'] is False
assert manifest['latest_date'] <= '2023-12-31'
assert manifest['rows'] == len(train)

cv_details = pd.read_csv(VALIDATION_DIR / 'fold_metrics.csv')
cv_summary = pd.read_csv(VALIDATION_DIR / 'grid_summary.csv')
cv_selected = pd.read_csv(VALIDATION_DIR / 'selected.csv')
print('Entorno de la corrida:', manifest['python'], 'scikit-learn', manifest['scikit_learn'])
print('Seleccionados por macro-F1 de validación (desviación entre años):')
display(cv_selected[['model', 'parameters', 'macro_f1_mean', 'macro_f1_std',
                     'validation_error_mean', 'train_error_mean']].round(6))
selected_params = {
    row.model: json.loads(row.parameters) for row in cv_selected.itertuples()
}
id3_gain_elegido = selected_params['ID3']['min_info_gain']
nb_m_elegido = selected_params['NB propio']['m']
nb_alpha_elegido = selected_params['CategoricalNB']['alpha']
rf_params_elegidos = selected_params['Random Forest']

id3_cv = cv_summary.loc[cv_summary.model.eq('ID3')].copy()
id3_cv['min_info_gain'] = id3_cv.parameters.map(lambda p: json.loads(p)['min_info_gain'])
nb_cv = cv_summary.loc[cv_summary.model.eq('NB propio')].copy()
nb_cv['m'] = nb_cv.parameters.map(lambda p: json.loads(p)['m'])
nb_cv['macro_f1_medio'] = nb_cv['macro_f1_mean']


## 9. Curvas y resultados de todas las configuraciones

Cada curva muestra medias por año. La tabla conserva el detalle de cada
hiperparámetro, y `cv_details` permite revisar los tres folds individualmente.
Las estrellas marcan el mejor macro-F1 dentro de cada panel; en Random Forest
la selección global compara **todas** las combinaciones. Las desviaciones entre
solo tres años no son intervalos de confianza. El desempeño utilizado para
seleccionar parámetros no sustituye la evaluación final reservada.


In [ ]:
from IPython.display import Image

for model_name in ['ID3', 'NB propio', 'CategoricalNB', 'Random Forest']:
    print(model_name)
    display(cv_summary.loc[cv_summary.model.eq(model_name),
        ['parameters', 'train_error_mean', 'validation_error_mean',
         'train_macro_f1_mean', 'macro_f1_mean', 'macro_f1_std']].round(6))
for filename in ['id3.png', 'nb_propio.png', 'categorical_nb.png', 'random_forest.png']:
    display(Image(filename=str(VALIDATION_DIR / 'figures' / filename)))
print('Detalle anual de las configuraciones elegidas:')
display(cv_details.loc[cv_details.config_id.isin(cv_selected.config_id),
    ['model', 'parameters', 'validacion', 'train_error', 'validation_error', 'macro_f1']])


## 10. Experimentos acotados de atributos

Se comparan siete variantes declaradas antes de la corrida: actuales; +puntos;
+diferencia de gol; +ambos; +tasas históricas de empate; sin historial local;
sin historial H2H del local. Los cuatro modelos mantienen los hiperparámetros
seleccionados anteriormente. Cada variante usa los mismos folds anuales y un
preprocesamiento nuevo por fold. Las tasas de empate solo usan fechas anteriores.

El diagnóstico «nunca E» restringe las predicciones a L/V, usando probabilidades
(o conteos del nodo en ID3). Se conservan todos los empates reales y las tres
clases E/L/V al calcular macro-F1; no se entrena un clasificador binario.
El diagnóstico no participa en la selección principal de atributos.

Selección: macro-F1 medio anual; empate exacto -> menos atributos, luego orden
del plan. Reutilizar validación para decisiones sucesivas puede introducir
optimismo. No se ha ejecutado el reajuste ni la evaluación final.

Plan: `docs/feature_experiment_plan.md`. Resultados: `docs/feature_findings.md`.
Reproducción: `python3.12 scripts/run_feature_experiments.py`.


In [ ]:
FEATURE_RESULTS = ROOT / 'results/feature_experiments'
feature_manifest = json.loads((FEATURE_RESULTS / 'manifest.json').read_text())
for relative, expected in feature_manifest['implementation_sha256'].items():
    assert hashlib.sha256((ROOT / relative).read_bytes()).hexdigest() == expected, relative
assert feature_manifest['source_sha256'] == hashlib.sha256(RAW.read_bytes()).hexdigest()
assert feature_manifest['test_evaluated'] is False
assert feature_manifest['latest_date'] <= '2023-12-31'
feature_summary = pd.read_csv(FEATURE_RESULTS / 'summary.csv')
feature_selected = pd.read_csv(FEATURE_RESULTS / 'selected.csv')
print('Selección de atributos por modelo (excluye el diagnóstico):')
display(feature_selected[['model','variant','parameters','accuracy_mean',
                          'macro_f1_mean','macro_f1_std']].round(6))
print('Todas las variantes:')
display(feature_summary.loc[feature_summary.decision.eq('three_class'),
    ['model','variant','accuracy_mean','macro_f1_mean','draw_recall_mean']].round(6))
paired = feature_summary.pivot(index=['model','variant'], columns='decision',
                              values=['accuracy_mean','macro_f1_mean'])
for metric in ['accuracy_mean','macro_f1_mean']:
    paired[(metric,'delta_never_draw')] = paired[(metric,'never_draw')] - paired[(metric,'three_class')]
print('Diagnóstico sin predecir E; macro-F1 siempre sobre las tres clases:')
display(paired.round(6))
display(Image(filename=str(FEATURE_RESULTS / 'accuracy_macro_f1.png')))


## 11. Evaluación final reservada (desactivada)

La selección temporal y los experimentos de atributos terminan aquí.
Las celdas legadas siguientes conservan diagnósticos de la configuración original
de seis tasas. Antes de una evaluación final habrá que adaptarlas a las variantes
elegidas en `feature_selected`. En esta etapa permanecen desactivadas y no
producen predicciones ni métricas sobre 2024–2025.


In [ ]:
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    all_matches = load_clean_matches(RAW)
    final_features = build_causal_match_features(all_matches)
    final_train, test = temporal_holdout(final_features)
    pd.testing.assert_frame_equal(train, final_train)
    X_test = discretizer.transform(test[NUMERIC_FEATURES])


def nuevo_discretizador_nb():
    return new_discretizer()


### 8. Entrenamiento del arbol

El procedimiento es recursivo y avaro:

1. se calcula la ganancia de todos los atributos disponibles en el nodo;
2. se parte por el de mayor ganancia (una rama por categoria);
3. cada atributo se usa **a lo sumo una vez por rama**;
4. se detiene cuando un nodo es puro, no quedan atributos, o la mejor ganancia no supera `min_info_gain`.

Se reajusta con el umbral elegido por macro-F1 en los folds comunes; se observan profundidad e importancias sin cambiar el umbral a partir de test.

In [ ]:
if RUN_FINAL_TEST:
    arbol_id3 = ID3(min_info_gain=id3_gain_elegido)
    arbol_id3.fit(X_train, train['winner'])
    
    print('Profundidad maxima:', arbol_id3.get_depth())
    print('Cantidad de hojas:', arbol_id3.get_n_leaves())
    print('\nImportancia de atributos (ganancia acumulada normalizada):')
    importancias = pd.DataFrame({
        'atributo': NUMERIC_FEATURES,
        'importancia': arbol_id3.feature_importances_,
    }).sort_values('importancia', ascending=False)
    display(importancias)


### 9. Evaluacion sobre test (2024-2025)

El test no ajustó los cuantiles ni seleccionó el umbral del árbol. Sus historiales sí incorporan resultados de fechas anteriores, como en todos los modelos. Medimos:

- **accuracy**: proporcion de predicciones correctas;
- **macro-F1**: media aritmetica del F1 de cada clase (no favorece a la clase mayoritaria);
- **reporte por clase** (precision, recall, F1) coincidente con la matriz de confusion.

In [ ]:
if RUN_FINAL_TEST:
    y_test = test['winner'].to_numpy()
    y_pred = arbol_id3.predict(X_test)
    
    print('Accuracy:', round(accuracy_score(y_test, y_pred), 4))
    print('Macro-F1:', round(f1_score(y_test, y_pred, average='macro', zero_division=0), 4))
    print(classification_report(y_test, y_pred, digits=4, zero_division=0))
    
    matrix = confusion_matrix(y_test, y_pred, labels=CLASSES)
    fig, axis = plt.subplots(figsize=(5, 4))
    image = axis.imshow(matrix, cmap='Blues')
    axis.set_xticks(range(3), CLASSES)
    axis.set_yticks(range(3), CLASSES)
    axis.set_xlabel('prediccion')
    axis.set_ylabel('real')
    for row in range(3):
        for column in range(3):
            axis.text(column, row, str(matrix[row, column]), ha='center', va='center')
    plt.show()


### 10. Ejemplos concretos: donde acierta y donde falla

Vemos partidos reales de test con la prediccion y el resultado. La columna `acierta` indica si ID3 acerto.

In [ ]:
if RUN_FINAL_TEST:
    predicciones = test[['date', 'home', 'away', 'winner']].copy()
    predicciones['prediccion_id3'] = y_pred
    predicciones['acierta'] = predicciones['prediccion_id3'] == predicciones['winner']
    display(predicciones.sample(8, random_state=RANDOM_STATE))
    print('Aciertos totales:', int(predicciones['acierta'].sum()), '/', len(predicciones))
    
    empates_reales = predicciones[predicciones['winner'] == 'E']
    print('Sobre los', len(empates_reales), 'empates reales de test, '
          f'ID3 predijo exactamente empate en {(empates_reales["prediccion_id3"] == "E").sum()} '
          f'({(empates_reales["prediccion_id3"] == "E").mean():.1%}).')
    display(empates_reales.head(3))


### Comparadores con parámetros ya seleccionados

Se conservan los árboles de scikit-learn con códigos y con tasas continuas.
CategoricalNB y Random Forest utilizan exclusivamente las configuraciones elegidas
por validación. El baseline comparte la política de información histórica.
Esta sección está desactivada hasta la etapa de evaluación final.


In [ ]:
if RUN_FINAL_TEST:
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.naive_bayes import CategoricalNB
    
    X_train_raw = train[NUMERIC_FEATURES].to_numpy()
    X_test_raw = test[NUMERIC_FEATURES].to_numpy()
    y_train = train['winner'].to_numpy()
    competidores = {
        'sklearn DT (mismo input)': DecisionTreeClassifier(criterion='entropy', random_state=42),
        'sklearn DT (tasas crudas)': DecisionTreeClassifier(criterion='entropy', random_state=42),
        'sklearn CategoricalNB': CategoricalNB(alpha=nb_alpha_elegido, min_categories=4),
        'sklearn RandomForest (crudo)': RandomForestClassifier(
            n_estimators=300, class_weight='balanced_subsample', random_state=42,
            **rf_params_elegidos),
    }
    filas = [{'modelo':'ID3 seleccionado', 'accuracy':accuracy_score(y_test,y_pred),
              'macro_f1':f1_score(y_test,y_pred,labels=CLASSES,average='macro',zero_division=0)}]
    for nombre, modelo in competidores.items():
        usa_codigos = 'mismo input' in nombre or 'CategoricalNB' in nombre
        modelo.fit(X_train if usa_codigos else X_train_raw, y_train)
        predicha = modelo.predict(X_test if usa_codigos else X_test_raw)
        filas.append({'modelo':nombre, 'accuracy':accuracy_score(y_test,predicha),
                      'macro_f1':f1_score(y_test,predicha,labels=CLASSES,average='macro',zero_division=0)})
    base_10 = TenYearWinRateClassifier().fit(train[BASELINE_FEATURES],train['winner'])
    pred_base = base_10.predict(test[BASELINE_FEATURES])
    filas.append({'modelo':'Base 10 años','accuracy':accuracy_score(y_test,pred_base),
                  'macro_f1':f1_score(y_test,pred_base,labels=CLASSES,average='macro',zero_division=0)})
    display(pd.DataFrame(filas))


### 13. Efectividad del algoritmo

Para interpretar los numeros hay que compararlos con los baselines: **predecir siempre `L`** (regla fija de referencia) y la **base de 10 anios** (el clasificador de la letra, sección 12). Tambien revisamos la distribucion de predicciones y el recall por clase.

In [ ]:
if RUN_FINAL_TEST:
    baseline_trivial = float((test['winner'] == 'L').mean())
    print('Accuracy de predecir siempre L:', round(baseline_trivial, 4))
    print('Accuracy de la base 10 anios:   ', round(accuracy_score(y_test, pred_base), 4))
    print(f'Accuracy del ID3 (gain={id3_gain_elegido:g}):', round(accuracy_score(y_test, y_pred), 4))
    print('\nDistribucion de predicciones del ID3 en test:')
    display(
        pd.Series(y_pred)
        .value_counts()
        .reindex(CLASSES)
        .rename('predicciones')
        .to_frame()
    )
    
    reporte = classification_report(
        y_test, y_pred, output_dict=True, zero_division=0
    )
    print('Precision/Recall/F1 por clase:')
    for clase in CLASSES:
        valor = reporte[clase]
        print(f'  {clase}: precision={valor["precision"]:.3f} '
              f'recall={valor["recall"]:.3f} f1={valor["f1-score"]:.3f}')


### Análisis final pendiente

Las métricas sobre test, errores por clase y ejemplos se analizarán cuando se
habilite la evaluación final. Los hallazgos actuales corresponden únicamente a
validación temporal y se documentan en `docs/validation_findings.md`.


### 14.2. Reajuste con todo el historico hasta 2023

Con `m` ya fijado, ajustamos un discretizador nuevo y un Naive Bayes nuevo con **todo train**. Usamos exclusivamente las seis columnas de `NUMERIC_FEATURES`; `winner` se pasa solo como etiqueta. El test 2024-2025 se transforma despues con los cortes aprendidos en train, sin volver a ajustarlos.

In [ ]:
if RUN_FINAL_TEST:
    # Ya elegido m, aprende las tablas definitivas usando todo train.
    nb_discretizador = nuevo_discretizador_nb()
    nb_X_train = nb_discretizador.fit_transform(train[NUMERIC_FEATURES])
    nb_modelo = MEstimateCategoricalNB(m=nb_m_elegido)
    nb_modelo.fit(nb_X_train, train['winner'])
    
    print(f'Naive Bayes ajustado con m={nb_m_elegido:g} '
          f'y {len(train):,} partidos hasta 2023.')
    display(pd.DataFrame({
        'atributo': NUMERIC_FEATURES,
        'cortes': [nb_discretizador.numeric_edges_[c].tolist()
                   for c in NUMERIC_FEATURES],
    }))


### 14.3. Evaluacion unica sobre test (2024-2025)

Obtenemos una sola tanda de predicciones finales de Naive Bayes. Todas las metricas, la distribucion y los ejemplos siguientes reutilizan ese resultado; no se vuelve a elegir `m` a partir del test. Mostramos accuracy, macro-F1, reporte por clase y matriz de confusion en el orden `E, L, V`, con filas reales y columnas predichas.

In [ ]:
if RUN_FINAL_TEST:
    # Test solo se transforma y predice: aqui no se ajusta ningun parametro.
    nb_X_test = nb_discretizador.transform(test[NUMERIC_FEATURES])
    nb_y_pred = nb_modelo.predict(nb_X_test)
    nb_accuracy = accuracy_score(y_test, nb_y_pred)
    nb_macro_f1 = f1_score(
        y_test, nb_y_pred, labels=CLASSES, average='macro', zero_division=0
    )
    nb_reporte = classification_report(
        y_test, nb_y_pred, labels=CLASSES, output_dict=True, zero_division=0
    )
    
    print('Accuracy:', round(nb_accuracy, 4))
    print('Macro-F1:', round(nb_macro_f1, 4))
    print(classification_report(
        y_test, nb_y_pred, labels=CLASSES, digits=4, zero_division=0
    ))
    
    nb_matrix = confusion_matrix(y_test, nb_y_pred, labels=CLASSES)
    nb_fig, nb_axis = plt.subplots(figsize=(5, 4))
    nb_axis.imshow(nb_matrix, cmap='Blues')
    nb_axis.set_xticks(range(len(CLASSES)), CLASSES)
    nb_axis.set_yticks(range(len(CLASSES)), CLASSES)
    nb_axis.set_xlabel('prediccion')
    nb_axis.set_ylabel('real')
    nb_axis.set_title(f'Naive Bayes (m={nb_m_elegido:g})')
    for nb_fila in range(len(CLASSES)):
        for nb_columna in range(len(CLASSES)):
            nb_axis.text(
                nb_columna, nb_fila, str(nb_matrix[nb_fila, nb_columna]),
                ha='center', va='center',
            )
    plt.show()
    
    nb_distribucion = pd.Series(nb_y_pred).value_counts().reindex(
        CLASSES, fill_value=0
    ).rename('predicciones').to_frame()
    nb_distribucion['proporcion'] = nb_distribucion['predicciones'] / len(y_test)
    nb_distribucion['reales'] = pd.Series(y_test).value_counts().reindex(
        CLASSES, fill_value=0
    )
    print('Distribucion de predicciones de Naive Bayes en test:')
    display(nb_distribucion)


### 14.4. Partidos reales: aciertos y errores

Tomamos hasta cuatro aciertos y cuatro errores, con la semilla existente para que la muestra sea reproducible. Los nombres, las fechas y la etiqueta real se muestran para interpretar los resultados; no se agregan como entradas al modelo. Revisamos tambien los primeros empates reales del test.

In [ ]:
if RUN_FINAL_TEST:
    nb_predicciones = test[['date', 'home', 'away', 'winner']].copy()
    nb_predicciones['prediccion_nb'] = nb_y_pred
    nb_predicciones['acierta'] = nb_predicciones['prediccion_nb'].eq(
        nb_predicciones['winner']
    )
    for nb_acierta, nb_titulo in [(True, 'Aciertos'), (False, 'Errores')]:
        nb_grupo = nb_predicciones.loc[nb_predicciones['acierta'] == nb_acierta]
        print(f'{nb_titulo}: {len(nb_grupo)} de {len(nb_predicciones)} partidos')
        display(nb_grupo.sample(
            n=min(4, len(nb_grupo)), random_state=RANDOM_STATE
        ).sort_values('date'))
    
    print('Ejemplos de empates reales:')
    display(nb_predicciones.loc[nb_predicciones['winner'] == 'E'].head(3))


### 14.5. Comparación con ID3

Se reutilizan las predicciones del ID3 seleccionado en los folds comunes y
reajustado con todo train. Ambos modelos comparten la política de datos,
historiales y validación. Esta tabla no interviene en seleccionar parámetros.


In [ ]:
if RUN_FINAL_TEST:
    nb_comparacion = pd.DataFrame([
        {
            'modelo': f'ID3 (gain={id3_gain_elegido:g})',
            'accuracy': accuracy_score(y_test, y_pred),
            'macro_f1': f1_score(
                y_test, y_pred, labels=CLASSES, average='macro', zero_division=0
            ),
        },
        {
            'modelo': f'Naive Bayes propio (m={nb_m_elegido:g})',
            'accuracy': nb_accuracy,
            'macro_f1': nb_macro_f1,
        },
    ])
    display(nb_comparacion.round(4))


### 14.6. Como interpretar los resultados

**Empates (`E`).** La distribucion de predicciones y el recall muestran cuantos empates se detectan y cuantos se asignan a `L` o `V`. Las seis tasas describen victorias, pero no distinguen explicitamente un empate de una derrota dentro de los partidos no ganados: esto limita la informacion disponible para separar `E`.

**Efecto de m.** El rango de macro-F1 medio en validacion muestra la sensibilidad a los valores probados. Aumentar `m` acerca las probabilidades condicionales al prior uniforme; no agrega informacion sobre empates ni balancea automaticamente las clases. La seleccion se limita a esta grilla y estos tres bloques, y no garantiza el mejor resultado en otros periodos.

**Limitaciones.** Varias tasas resumen historiales superpuestos, por lo que la independencia condicional de Naive Bayes es una aproximacion. Ademas, los tres bines comprimen las tasas y el neutro `0.5` coincide con valores que pueden tener equipos con historial. Estas limitaciones ayudan a interpretar las diferencias con ID3, pero las metricas por si solas no demuestran cual es la causa de cada error.

In [ ]:
if RUN_FINAL_TEST:
    nb_empates = nb_reporte['E']
    print(
        f'Empates: {int(nb_empates["support"])} reales, '
        f'{int(nb_distribucion.loc["E", "predicciones"])} predichos y '
        f'{int(nb_matrix[CLASSES.index("E"), CLASSES.index("E")])} acertados. '
        f'Precision={nb_empates["precision"]:.4f}, '
        f'recall={nb_empates["recall"]:.4f}, F1={nb_empates["f1-score"]:.4f}.'
    )
    if nb_empates['recall'] < min(nb_reporte[c]['recall'] for c in ['L', 'V']):
        print('E es la clase con menor recall: se recupera una proporcion '
              'menor de empates que de victorias locales o visitantes.')
    
    nb_rango_cv = nb_cv['macro_f1_medio'].max() - nb_cv['macro_f1_medio'].min()
    print(
        f'm elegido={nb_m_elegido:g}. El macro-F1 medio de validacion va de '
        f'{nb_cv["macro_f1_medio"].min():.6f} a '
        f'{nb_cv["macro_f1_medio"].max():.6f} '
        f'(diferencia={nb_rango_cv:.6f}).'
    )
    if nb_rango_cv == 0:
        print('Los valores de m empatan en macro-F1 medio; se eligio el menor.')
    print(
        'Diferencia Naive Bayes - ID3 en test: '
        f'accuracy={nb_accuracy - nb_comparacion.loc[0, "accuracy"]:+.4f}; '
        f'macro-F1={nb_macro_f1 - nb_comparacion.loc[0, "macro_f1"]:+.4f}. '
        'Estas diferencias describen el test y no se usaron para elegir m.'
    )


## Pendientes

La selección de hiperparámetros y la comparación acotada de atributos están
completas. La evaluación final requiere aplicar las configuraciones elegidas,
reajustarlas y analizar métricas por clase en una etapa posterior.
No se han cerrado conclusiones de test.
